# Atari DQN — Crazy Climber

**Before running:** `Runtime → Change runtime type → T4 GPU`

This notebook runs **3 hyperparameter configurations** to find the best setup for Crazy Climber.
All checkpoints and `metrics.csv` files are saved to Google Drive.

| Target | Score |
|--------|-------|
| Minimum — random agent | 10,781 |
| Good grade — linear agent | 23,411 |
| Bonus — DQN paper score | 114,103 |

In [ ]:
# ── Cell 1: Mount Google Drive ───────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/Atari-DQN'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'Drive mounted. Outputs -> {DRIVE_DIR}/results')

In [ ]:
# ── Cell 2: Clone project from GitHub ────────────────────────────────────────
# Edit REPO_URL to point at your repository before running.
REPO_URL = 'https://github.com/YOUR_USERNAME/Atari-DQN.git'  # <- change this

import os
import subprocess

REPO_DIR = '/content/Atari-DQN'

if os.path.exists(REPO_DIR):
    print('Repo exists — pulling latest changes...')
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)
else:
    print(f'Cloning {REPO_URL} ...')
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)

os.chdir(REPO_DIR)
print('cwd:', os.getcwd())

In [ ]:
# ── Cell 3: Install dependencies ─────────────────────────────────────────────
# Skip torch because Colab already provides it; reinstalling it often causes conflicts.
import subprocess

with open('requirements.txt', 'r', encoding='utf-8') as handle:
    packages = [
        line.strip()
        for line in handle
        if line.strip() and not line.lstrip().startswith('torch')
    ]

subprocess.run(['pip', 'install', '-q', *packages], check=True)
print('Dependencies installed.')

In [ ]:
# ── Cell 4: Verify setup ─────────────────────────────────────────────────────
import sys
sys.path.insert(0, '/content/Atari-DQN')

import torch, gymnasium
print(f'PyTorch  : {torch.__version__}')
print(f'CUDA     : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU      : {torch.cuda.get_device_name(0)}')
print(f'Gymnasium: {gymnasium.__version__}')

from src.config import Config
from src.train import train
print('Imports  : OK')

---
## Config 1 — Adam, lr=1e-4, ε-decay=500k steps
Fast epsilon decay: tests whether quicker exploitation benefits Crazy Climber.

In [ ]:
cfg1 = Config(
    run_id='config1_adam_fast',
    seed=42,
    total_steps=2_000_000,
    results_dir=f'{DRIVE_DIR}/results',
    optimizer='adam',
    lr=1e-4,
    epsilon_decay_steps=500_000,
    epsilon_end=0.1,
    eval_episodes=5,
)
print('Starting Config 1: Adam lr=1e-4, epsilon_decay=500k')
rewards1 = train(cfg1)
print(f'Config 1 done. Final mean reward: {rewards1[-1]:.1f}')

---
## Config 2 — RMSprop, lr=2.5e-4, ε-decay=1M steps
Paper-style optimizer with a standard exploration schedule.

In [ ]:
cfg2 = Config(
    run_id='config2_rmsprop_standard',
    seed=42,
    total_steps=2_000_000,
    results_dir=f'{DRIVE_DIR}/results',
    optimizer='rmsprop',
    lr=2.5e-4,
    rmsprop_alpha=0.95,
    rmsprop_eps=0.01,
    epsilon_decay_steps=1_000_000,
    epsilon_end=0.1,
    eval_episodes=5,
)
print('Starting Config 2: RMSprop lr=2.5e-4, epsilon_decay=1M')
rewards2 = train(cfg2)
print(f'Config 2 done. Final mean reward: {rewards2[-1]:.1f}')

---
## Config 3 — Adam, lr=5e-5, batch=64, ε-decay=750k steps
Slower, more stable learning with a larger batch size.

In [ ]:
cfg3 = Config(
    run_id='config3_adam_stable',
    seed=42,
    total_steps=2_000_000,
    results_dir=f'{DRIVE_DIR}/results',
    optimizer='adam',
    lr=5e-5,
    batch_size=64,
    epsilon_decay_steps=750_000,
    epsilon_end=0.1,
    eval_episodes=5,
)
print('Starting Config 3: Adam lr=5e-5, batch=64, epsilon_decay=750k')
rewards3 = train(cfg3)
print(f'Config 3 done. Final mean reward: {rewards3[-1]:.1f}')

---
## Results — Compare all 3 configurations
Reads metrics CSVs from Drive — safe to re-run without re-training.

In [ ]:
import matplotlib.pyplot as plt, csv, os

run_configs = [
    ('config1_adam_fast',        'Config 1: Adam lr=1e-4, decay=500k',    'steelblue'),
    ('config2_rmsprop_standard', 'Config 2: RMSprop lr=2.5e-4, decay=1M', 'darkorange'),
    ('config3_adam_stable',      'Config 3: Adam lr=5e-5, batch=64',       'seagreen'),
]

fig, ax = plt.subplots(figsize=(12, 5))
final_rewards = {}

for run_id, label, color in run_configs:
    csv_path = os.path.join(DRIVE_DIR, 'results', run_id, 'metrics.csv')
    steps, means = [], []
    with open(csv_path, newline='') as f:
        for row in csv.DictReader(f):
            steps.append(int(row['step']))
            means.append(float(row['mean_eval_reward']))
    ax.plot(steps, means, label=label, color=color, linewidth=1.5)
    final_rewards[run_id] = means[-1]

ax.axhline(10_781, color='red',    linestyle='--', linewidth=1, alpha=0.7, label='Min target (10,781)')
ax.axhline(23_411, color='orange', linestyle='--', linewidth=1, alpha=0.7, label='Good grade (23,411)')
ax.set_xlabel('Training steps')
ax.set_ylabel('Mean evaluation reward')
ax.set_title('Crazy Climber — Config comparison (2M steps each)')
ax.legend(loc='upper left')
ax.grid(True, linestyle='--', alpha=0.4)
fig.tight_layout()

plot_path = os.path.join(DRIVE_DIR, 'results', 'config_comparison.png')
fig.savefig(plot_path, dpi=150)
plt.show()
print(f'Plot saved to {plot_path}')

best_run = max(final_rewards, key=final_rewards.get)
print()
print('--- Final mean rewards ---')
for run_id, label, _ in run_configs:
    marker = '  <-- best' if run_id == best_run else ''
    print(f'  {label}: {final_rewards[run_id]:.1f}{marker}')